In [1]:
import math

import torch
from torch import nn
from d2l import torch as d2l

In [2]:
# Position-Wise Feed-Forward Network

class PositionWiseFFN(nn.Module):
    
    def __init__(
        self,
        ffn_num_hiddens: int,
        ffn_num_outputs: int,
    ) -> None:
        super().__init__()
        
        self.dense1 = nn.LazyLinear(
            out_features=ffn_num_hiddens,
        )
        
        self.relu = nn.ReLU()
        
        self.dense2 = nn.LazyLinear(
            out_features=ffn_num_outputs,
        )
        
        
    def forward(
        self,
        X: torch.Tensor,
    ) -> torch.Tensor:

        return self.dense2(
            self.relu(
                self.dense1(X)
            )
        )

In [3]:
# AddNorm

class AddNorm(nn.Module):

    def __init__(
        self,
        norm_shape: int,
        dropout: float,
    ) -> None:
        super().__init__()

        self.dropout = nn.Dropout(
            p=dropout,
        )

        self.layer_norm = nn.LayerNorm(
            normalized_shape=norm_shape,
        )

    def forward(
        self,
        X: torch.Tensor,
        Y: torch.Tensor,
    ) -> torch.Tensor:

        if X.shape != Y.shape:
            raise ValueError(
                "X and Y must have the same shape."
            )

        return self.layer_norm(
            X + self.dropout(Y)
        )

In [4]:
# Transformer Encoder Block

class TransformerEncoderBlock(nn.Module):
    
    def __init__(
        self,
        num_hiddens: int,
        ffn_num_hiddens: int,
        num_heads: int,
        dropout: float,
        use_bias: bool = False,
    ) -> None:
        super().__init__()
        
        self.attention = d2l.MultiHeadAttention(
            num_hiddens=num_hiddens,
            num_heads=num_heads,
            dropout=dropout,
            bias=use_bias,
        )
        
        self.add_norm1 = AddNorm(
            norm_shape=num_hiddens,
            dropout=dropout,
        )
        
        self.ffn = PositionWiseFFN(
            ffn_num_hiddens=ffn_num_hiddens,
            ffn_num_outputs=num_hiddens,
        )
        
        self.add_norm2 = AddNorm(
            norm_shape=num_hiddens,
            dropout=dropout,
        )
        
        
    def forward(
        self,
        X: torch.Tensor,
        valid_lens: torch.Tensor | None,
    ) -> torch.Tensor:
        
        # Encoder Self-Attention: Q = K = V =X
        # [B, T, D] -> [B, T, D]
        attention_output = self.attention(
            queries=X,
            keys=X,
            values=X,
            valid_lens=valid_lens,
        )
        
        # First Residual Connection + LayerNorm:
        # [B, T, D] + [B, T, D]
        Y = self.add_norm1(
            X,
            attention_output,
        )
        
        # Position-Wise FFN:
        # [B, T, D] -> [B, T, D]
        ffn_output = self.ffn(
            Y
        )
        
        # Second Residual Connection + LayerNorm
        return self.add_norm2(
            Y,
            ffn_output,
        )    
        

In [5]:
# Encoder Block shape 검증

batch_size = 2
num_steps = 100
num_hiddens = 24

num_heads = 8
ffn_num_hiddens = 48

# [B, T, D]
X = torch.ones(
    (
        batch_size,
        num_steps,
        num_hiddens,
    )
)

valid_lens = torch.tensor([
    3,
    2,
])

encoder_block = TransformerEncoderBlock(
    num_hiddens=num_hiddens,
    ffn_num_hiddens=ffn_num_hiddens,
    num_heads=num_heads,
    dropout=0.5,
)

encoder_block.eval()

with torch.no_grad():
    block_output = encoder_block(
        X, 
        valid_lens,
    )



print(
    "Input shape:",
    tuple(X.shape),
)

print(
    "Block output shape:",
    tuple(block_output.shape),
)

d2l.check_shape(
    block_output,
    X.shape,
)

Input shape: (2, 100, 24)
Block output shape: (2, 100, 24)


In [6]:
# Transformer Encoder

class TransformerEncoder(d2l.Encoder):
    
    def __init__(
        self,
        vocab_size: int,
        num_hiddens: int,
        ffn_num_hiddens: int,
        num_heads: int,
        num_blocks: int,
        dropout: float,
        use_bias: bool = False,
    ) -> None:
        super().__init__()
        
        self.num_hiddens = num_hiddens

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=num_hiddens,
        )

        self.positional_encoding = d2l.PositionalEncoding(
            num_hiddens=num_hiddens,
            dropout=dropout,
        )

        self.blocks = nn.Sequential(
            *[
                TransformerEncoderBlock(
                    num_hiddens=num_hiddens,
                    ffn_num_hiddens=ffn_num_hiddens,
                    num_heads=num_heads,
                    dropout=dropout,
                    use_bias=use_bias,
                )
                for _ in range(num_blocks)
            ]
        )
        
        self.attention_weights: list[
            torch.Tensor
        ] = []
        
        
    def forward(
        self,
        X: torch.Tensor,
        valid_lens: torch.Tensor | None,
        *args: object,
    ) -> torch.Tensor:
        
        # Embedding: [B, T] -> √D x [B, T, D]
        X = (
            self.embedding(X)
            * math.sqrt(
                self.num_hiddens 
            )
        )
        
        # Positional Encoding:
        X = self.positional_encoding(
            X
        )
        
        self.attention_weights = []
        
        for block in self.blocks:
            if not isinstance(
                block,
                TransformerEncoderBlock,
            ):
                raise TypeError(
                    "Every encoder block must be "
                    "a TransformerEncoderBlock."
                )
                
            # Forward "X" into TransformerEncoderBlock
            X = block(
                X,
                valid_lens,
            )
            
            weights = getattr(
                block.attention.attention,
                "attention_weights",
                None,
            )

            if not isinstance(
                weights,
                torch.Tensor,
            ):
                raise RuntimeError(
                    "Attention weights were not computed."
                )
                
            self.attention_weights.append(
                weights
            )
            
        return X

In [7]:
# 전체 Transformer Encoder 검증

vocab_size = 200
num_hiddens = 24
ffn_num_hiddens = 48

num_heads = 8
num_blocks = 2

batch_size = 2
num_steps = 100

encoder = TransformerEncoder(
    vocab_size=vocab_size,
    num_hiddens=num_hiddens,
    ffn_num_hiddens=ffn_num_hiddens,
    num_heads=num_heads,
    num_blocks=num_blocks,
    dropout=0.5,
)

encoder.eval()

# [B, T] = [2, 100]
tokens = torch.ones(
    (
        batch_size,
        num_steps,
    ),
    dtype=torch.long,
)

valid_lens = torch.tensor([
    3,
    2,
])


with torch.no_grad():
    # [B, T, H] = [2, 100, 24]
    encoder_output = encoder(
        tokens,
        valid_lens,
    )
    

print(
    "Token shape:",
    tuple(tokens.shape),
)

print(
    "Encoder output shape:",
    tuple(encoder_output.shape),
)

# Encoder Block이 두 개이므로 Attention Weight도 두 개 저장
assert (
    len(
        encoder.attention_weights
    )
    == num_blocks
)


for (
    block_index,
    weights,
) in enumerate(
    encoder.attention_weights,
):

    print(
        f"Block {block_index + 1} "
        f"attention weights:",
        tuple(weights.shape),
    )

    # d2l.MultiHeadAttention은 Batch와 Head를 결합:
    # [B × Head, Query, Key] = [2 × 8, 100, 100]
    d2l.check_shape(
        weights,
        (
            batch_size * num_heads,
            num_steps,
            num_steps,
        ),
    )

Token shape: (2, 100)
Encoder output shape: (2, 100, 24)
Block 1 attention weights: (16, 100, 100)
Block 2 attention weights: (16, 100, 100)
